In [ ]:
import requests
import pandas as pd
import time

# URL base
url = "https://www.drogariasaopaulo.com.br/api/catalog_system/pub/products/search"

# Lista de SKUs
sku_list = [
"629138","628425","627526","578525","567337","548014","542725","529346",
"515167","511102","510890","387975","360201","295710","290220","288292",
"222011","219789","204005","195480","183130","165760","68209","51853",
"907081","906476","906425","898392","896888","896870","893552","886513",
"883530","883166","883131","875198","870331","868744","864668","862401",
"861162","861111","860816","860174","855499","855308","852414","837113",
"497398","476455","474193","473855","465100","430838","423866","385948",
"385034","360325","288284","271683","222470","208949","203548","194247",
"183377","165514","145971","145807","134597","108618","108588","95249",
"89010","87300","80900","73113","41858","41580","33804","28916","27537",
"906450","906433","906417","901490","900753","897205","897060","897051",
"896896","896470","896233","896195","896152","895970","895962"
]

todos = []

# Loop por SKU individual
for i, sku in enumerate(sku_list, 1):
    params = {
        "_from": 0,
        "_to": 49,  # só precisamos de 1 produto por vez
        "fq": f"skuId:{sku}"  # apenas 1 SKU por requisição
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if data:  # se a API retornou algum produto
            todos.extend(data)
        
        print(f"SKU {i}/{len(sku_list)} coletado: {sku} → Total produtos: {len(todos)}")
        time.sleep(0.3)  # evita bloqueio do site
    
    except requests.HTTPError as e:
        print(f"Erro no SKU {sku}: {e}")
        time.sleep(1)  # espera um pouco e continua

# Transformar em DataFrame completo
df = pd.DataFrame(todos)

# Criar DataFrame somente com campos relevantes
df_clean = pd.DataFrame([
    {
        "id": produto.get("productId"),
        "nome": produto.get("productName"),
        "sku": produto.get("items")[0]["itemId"] if produto.get("items") else None,
        "preco": produto.get("items")[0]["sellers"][0]["commertialOffer"]["Price"] 
                 if produto.get("items") else None
    }
    for produto in todos
])

sku_retornados = [produto["items"][0]["itemId"] for produto in todos if produto.get("items")]
skus_nao_encontrados = [sku for sku in sku_list if sku not in sku_retornados]

print("SKUs não encontrados / ausentes na API:")
print(skus_nao_encontrados)

# Salvar em CSV
df_clean.to_csv("produtos_drogariasp.csv", index=False)
print("CSV salvo com sucesso")

In [ ]:
display(df_clean)
